# Proyecto 1 — Clasificación de Contaminaciones en Línea de Producción

Entrena y compara cuatro algoritmos de clasificación (Árbol de Decisión, Naive Bayes, KNN, SVM) sobre imágenes binarizadas 128×128 para detectar granos de arroz en línea de producción.

Ver el `README.md` del repositorio para instrucciones de uso, formato del dataset e inferencia.


---
## 0. Importaciones


In [ ]:
# Google Colab ya incluye todas estas librerías
# Si corrés localmente descomentá la siguiente línea:
# !pip install scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import joblib
import time

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedShuffleSplit,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)

warnings.filterwarnings("ignore")
np.random.seed(42)
print("Librerias importadas correctamente")

---
## 1. Configuración

Completá tus datos y ajustá los parámetros si es necesario. Luego subí tus CSVs cuando la celda lo solicite.


In [ ]:
from google.colab import files

# ── Subir uno o varios CSV desde tu computadora ───────────
print("Selecciona tus archivos CSV (Ctrl+Click para seleccionar varios)")
uploaded = files.upload()
ARCHIVOS_CSV = list(uploaded.keys())
print("")
print(str(len(ARCHIVOS_CSV)) + " archivo(s) cargado(s): " + str(ARCHIVOS_CSV))

# ── Datos de identificación ───────────────────────────────
CARNE    = "X00000"              # <- Cambiar por tu carne
NOMBRE   = "NombreEstudiante"    # <- Cambiar por tu nombre
APELLIDO = "ApellidoEstudiante"  # <- Cambiar por tu apellido

# ── Parámetros del experimento ────────────────────────────
TEST_SIZE      = 0.2   # 20% para prueba
N_REPETICIONES = 20    # repeticiones de evaluación robusta
CV_FOLDS       = 5     # pliegues para GridSearchCV
RANDOM_STATE   = 42

# Nombre de la columna de etiqueta en el CSV
COLUMNA_ETIQUETA = "etiqueta"

print("")
print("Estudiante  : " + NOMBRE + " " + APELLIDO + " (" + CARNE + ")")
print("Split       : " + str(int((1 - TEST_SIZE) * 100)) + "% entrenamiento / " + str(int(TEST_SIZE * 100)) + "% prueba")
print("Repeticiones: " + str(N_REPETICIONES))

---
## 2. Carga del dataset

Acepta uno o varios CSVs. Si subís más de uno se combinan automáticamente.


In [ ]:
def cargar_un_csv(ruta_csv):
    """
    Carga un CSV con encabezado pixel_1...pixel_16384, etiqueta.
    Recupera todas las filas como datos (ninguna se pierde como encabezado).
    """
    df = pd.read_csv(ruta_csv)

    if COLUMNA_ETIQUETA not in df.columns:
        # Fallback: sin encabezado reconocido, tratar primera fila como dato
        df = pd.read_csv(ruta_csv, header=None)
        col_etiqueta = df.columns[-1]
        print("  Sin encabezado reconocido -> header=None, etiqueta = col " + str(col_etiqueta))
    else:
        col_etiqueta = COLUMNA_ETIQUETA

    df = df.apply(pd.to_numeric, errors="coerce")
    filas_antes = len(df)
    df = df.dropna()
    if len(df) < filas_antes:
        print("  Filas descartadas por datos no numericos: " + str(filas_antes - len(df)))

    X = df.drop(columns=[col_etiqueta]).values.astype(int)
    y = df[col_etiqueta].values.astype(int)
    return X, y


def cargar_datasets(lista_csv):
    """Carga y combina múltiples CSVs en un único dataset."""
    Xs, ys = [], []
    print("Cargando " + str(len(lista_csv)) + " archivo(s)...")
    print("")

    for ruta in lista_csv:
        print("Archivo: " + ruta)
        X_i, y_i = cargar_un_csv(ruta)

        if X_i.shape[1] != 128 * 128:
            print("  ERROR: se esperaban " + str(128*128) + " pixeles, tiene " + str(X_i.shape[1]) + ". Omitido.")
            continue

        print("  " + str(X_i.shape[0]) + " imagenes  (positivos=" + str(y_i.sum()) + ", negativos=" + str((y_i==0).sum()) + ")")
        Xs.append(X_i)
        ys.append(y_i)

    if not Xs:
        raise ValueError("No se pudo cargar ningun CSV valido.")

    X_total = np.vstack(Xs)
    y_total = np.concatenate(ys)
    return X_total, y_total


X, y = cargar_datasets(ARCHIVOS_CSV)

print("")
print("=" * 45)
print("DATASET COMBINADO")
print("=" * 45)
print("Total imagenes : " + str(X.shape[0]))
print("Pixeles/imagen : " + str(X.shape[1]) + " (128x128)")
print("Positivos      : " + str((y==1).sum()) + " (con arroz)")
print("Negativos      : " + str((y==0).sum()) + " (sin arroz)")

---
## 3. Exploración del dataset


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

clases  = ["Negativo (0)\nSin arroz", "Positivo (1)\nCon arroz"]
conteos = [(y==0).sum(), (y==1).sum()]
colores = ["#3498db", "#e74c3c"]

bars = axes[0].bar(clases, conteos, color=colores, edgecolor="white", linewidth=1.5, width=0.5)
axes[0].set_title("Distribucion de Clases", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Numero de imagenes")
for bar, cnt in zip(bars, conteos):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 str(cnt), ha="center", va="bottom", fontsize=13, fontweight="bold")
axes[0].set_ylim(0, max(conteos) * 1.25)
axes[0].grid(axis="y", alpha=0.3)

axes[1].pie(conteos, labels=clases, colors=colores, autopct="%1.1f%%",
            startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Proporcion de Clases", fontsize=13, fontweight="bold")

plt.suptitle("Analisis del Dataset", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("distribucion_clases.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: distribucion_clases.png")

In [ ]:
# Reconstruir y mostrar algunas imágenes del dataset
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Muestra del Dataset (reconstruidas desde pixeles)", fontsize=13, fontweight="bold")

idx_pos = np.where(y == 1)[0]
idx_neg = np.where(y == 0)[0]
n = min(5, len(idx_pos), len(idx_neg))

for i in range(n):
    img_pos = X[idx_pos[i]].reshape(128, 128)
    axes[0, i].imshow(img_pos, cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title("Positivo #" + str(idx_pos[i]) + "\n(con arroz)", fontsize=9, color="#e74c3c")
    axes[0, i].axis("off")

    img_neg = X[idx_neg[i]].reshape(128, 128)
    axes[1, i].imshow(img_neg, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title("Negativo #" + str(idx_neg[i]) + "\n(sin arroz)", fontsize=9, color="#3498db")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("muestra_imagenes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: muestra_imagenes.png")

---
## 4. División entrenamiento / prueba

Split estratificado para mantener la proporción de clases en ambos conjuntos. La evaluación repetida (sección 7) compensa la varianza de una única partición.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print("Division del dataset:")
print("  Entrenamiento: " + str(X_train.shape[0]) + " imagenes  (positivos=" + str(y_train.sum()) + ", negativos=" + str((y_train==0).sum()) + ")")
print("  Prueba       : " + str(X_test.shape[0])  + " imagenes  (positivos=" + str(y_test.sum())  + ", negativos=" + str((y_test==0).sum())  + ")")
print("")
print("Balance en entrenamiento: " + str(round(y_train.mean()*100, 1)) + "% positivos")
print("Balance en prueba       : " + str(round(y_test.mean()*100, 1))  + "% positivos")

---
## 5. Modelos e hiperparámetros

GridSearchCV busca la mejor combinación dentro de cada espacio definido.


In [ ]:
def obtener_modelos():
    return {
        "Arbol de Decision": (
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            {
                "max_depth":         [5, 10, 20, None],
                "min_samples_split": [2, 5, 10],
                "criterion":         ["gini", "entropy"],
            }
        ),
        "Naive Bayes": (
            GaussianNB(),
            {
                "var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3],
            }
        ),
        "KNN": (
            KNeighborsClassifier(),
            {
                "n_neighbors": [3, 5, 7, 11],
                "metric":      ["euclidean", "manhattan"],
            }
        ),
        "SVM": (
            SVC(random_state=RANDOM_STATE, probability=True),
            {
                "C":      [0.1, 1, 10],
                "kernel": ["linear", "rbf"],
            }
        ),
    }

modelos = obtener_modelos()
print(str(len(modelos)) + " algoritmos definidos:")
for nombre, (_, params) in modelos.items():
    n_comb = 1
    for v in params.values():
        n_comb *= len(v)
    print("  " + nombre + ": " + str(n_comb) + " combinaciones de hiperparametros")

---
## 6. Entrenamiento con GridSearchCV


In [ ]:
resultados_grid = {}
tiempos = {}

for nombre, (modelo, params) in modelos.items():
    print("=" * 55)
    print("Buscando hiperparametros: " + nombre)
    print("=" * 55)

    inicio = time.time()

    grid = GridSearchCV(
        estimator  = clone(modelo),
        param_grid = params,
        cv         = CV_FOLDS,
        scoring    = "accuracy",
        n_jobs     = -1,
        refit      = True,
    )
    grid.fit(X_train, y_train)

    t = time.time() - inicio
    tiempos[nombre] = t
    resultados_grid[nombre] = grid

    y_pred   = grid.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred)

    print("  Mejores hiperparametros : " + str(grid.best_params_))
    print("  Exactitud en CV (train) : " + str(round(grid.best_score_ * 100, 2)) + "%")
    print("  Exactitud en prueba     : " + str(round(acc_test * 100, 2)) + "%")
    print("  Tiempo de busqueda      : " + str(round(t, 1)) + "s")
    print("")

print("GridSearchCV completado para todos los modelos")

---
## 7. Evaluación robusta

Con datasets pequeños una sola partición puede ser engañosa. Se repite la evaluación `N_REPETICIONES` veces con particiones estratificadas distintas.


In [ ]:
def evaluar_repetido(grid_ajustado, X, y, n_reps=N_REPETICIONES):
    splitter = StratifiedShuffleSplit(
        n_splits     = n_reps,
        test_size    = TEST_SIZE,
        random_state = RANDOM_STATE,
    )
    accuracies, f1s = [], []

    for train_idx, test_idx in splitter.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        m = clone(grid_ajustado.best_estimator_)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)

        accuracies.append(accuracy_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred, zero_division=0))

    return accuracies, f1s


print("Evaluacion repetida (" + str(N_REPETICIONES) + " iteraciones por modelo)...")
print("")

eval_resultados = {}

for nombre, grid in resultados_grid.items():
    print("  Evaluando: " + nombre + "...", end=" ")
    accs, f1s = evaluar_repetido(grid, X, y)
    eval_resultados[nombre] = {
        "accuracies": accs,
        "f1_scores":  f1s,
        "acc_media":  np.mean(accs),
        "acc_std":    np.std(accs),
        "f1_media":   np.mean(f1s),
        "f1_std":     np.std(f1s),
        "best_params": grid.best_params_,
    }
    print("Acc=" + str(round(np.mean(accs), 4)) + " +/-" + str(round(np.std(accs), 4)) + "  F1=" + str(round(np.mean(f1s), 4)))

print("")
print("Evaluacion robusta completada")

---
## 8. Comparación de modelos


In [ ]:
filas = []
for nombre, r in eval_resultados.items():
    filas.append({
        "Modelo":         nombre,
        "Acc Media":      round(r["acc_media"], 4),
        "Acc Std":        round(r["acc_std"], 4),
        "Acc Min":        round(min(r["accuracies"]), 4),
        "Acc Max":        round(max(r["accuracies"]), 4),
        "F1 Media":       round(r["f1_media"], 4),
        "F1 Std":         round(r["f1_std"], 4),
        "Mejores Params": str(r["best_params"]),
    })

df_resumen = pd.DataFrame(filas).sort_values("Acc Media", ascending=False).reset_index(drop=True)
df_resumen.index += 1
mejor_nombre = df_resumen.iloc[0]["Modelo"]
mejor_acc    = df_resumen.iloc[0]["Acc Media"]
# (Los resultados completos se muestran en la Tabla Unificada más abajo)
print(f"Evaluación repetida completada. Mejor modelo provisional: {mejor_nombre} ({mejor_acc})")


In [ ]:
nombres    = list(eval_resultados.keys())
acc_medias = [eval_resultados[n]["acc_media"] for n in nombres]
acc_stds   = [eval_resultados[n]["acc_std"]   for n in nombres]
f1_medias  = [eval_resultados[n]["f1_media"]  for n in nombres]
f1_stds    = [eval_resultados[n]["f1_std"]    for n in nombres]

orden     = np.argsort(acc_medias)[::-1]
nombres_o = [nombres[i]    for i in orden]
acc_med_o = [acc_medias[i] for i in orden]
acc_std_o = [acc_stds[i]   for i in orden]
f1_med_o  = [f1_medias[i]  for i in orden]
f1_std_o  = [f1_stds[i]    for i in orden]

colores_bar = ["#27ae60", "#3498db", "#9b59b6", "#e67e22"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparacion de Modelos — Evaluacion Repetida (N=" + str(N_REPETICIONES) + ")",
             fontsize=13, fontweight="bold")

# Exactitud
bars = axes[0].bar(nombres_o, acc_med_o, yerr=acc_std_o, capsize=6,
                   color=colores_bar, edgecolor="white", linewidth=1.5)
axes[0].set_title("Exactitud Media +/- Std", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Exactitud")
axes[0].set_ylim(0, 1.15)
axes[0].axhline(y=0.5, color="red", linestyle="--", alpha=0.4, label="Azar (50%)")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
for bar, acc, std in zip(bars, acc_med_o, acc_std_o):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.02,
                 str(round(acc, 3)), ha="center", va="bottom", fontsize=10, fontweight="bold")

# F1
bars2 = axes[1].bar(nombres_o, f1_med_o, yerr=f1_std_o, capsize=6,
                    color=colores_bar, edgecolor="white", linewidth=1.5)
axes[1].set_title("F1-Score Medio +/- Std", fontsize=12, fontweight="bold")
axes[1].set_ylabel("F1-Score")
axes[1].set_ylim(0, 1.15)
axes[1].grid(axis="y", alpha=0.3)
for bar, f1, std in zip(bars2, f1_med_o, f1_std_o):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.02,
                 str(round(f1, 3)), ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("comparacion_modelos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: comparacion_modelos.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
datos_box = [eval_resultados[n]["accuracies"] for n in nombres_o]

bp = ax.boxplot(datos_box, labels=nombres_o, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], colores_bar):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Distribucion de Exactitudes — " + str(N_REPETICIONES) + " Repeticiones",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Exactitud por iteracion")
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.4, label="Azar (50%)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig("boxplot_exactitudes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: boxplot_exactitudes.png")
print("Los bigotes muestran la variabilidad. Box mas angosto = modelo mas estable.")

---
## 9. Matrices de confusión y métricas

La tabla unificada al final de esta sección agrupa todas las métricas: Accuracy, Precisión, Recall, F1, TP/TN/FP/FN, Acc CV y tiempo.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Matrices de Confusion (conjunto de prueba principal)",
             fontsize=12, fontweight="bold")

for ax, (nombre, grid) in zip(axes, resultados_grid.items()):
    y_pred = grid.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(
        confusion_matrix  = cm,
        display_labels    = ["Negativo\n(sin arroz)", "Positivo\n(con arroz)"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(nombre + "\nAcc=" + str(round(acc, 3)), fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("matrices_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: matrices_confusion.png")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# TABLA UNIFICADA DE MÉTRICAS — todos los parámetros de interés
# ════════════════════════════════════════════════════════════════════
from sklearn.metrics import precision_score, recall_score

filas = []
for nombre, grid in resultados_grid.items():
    r    = eval_resultados[nombre]
    y_pred = grid.predict(X_test)

    # métricas sobre el conjunto de prueba principal
    acc_test  = accuracy_score(y_test, y_pred)
    prec_test = precision_score(y_test, y_pred, zero_division=0)
    rec_test  = recall_score(y_test, y_pred, zero_division=0)
    f1_test   = f1_score(y_test, y_pred, zero_division=0)
    cm        = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (cm[0,0], 0, 0, cm[1,1])

    filas.append({
        "Modelo"           : nombre,
        # ── evaluación repetida ──
        "Acc Media (±Std)" : f"{r['acc_media']:.4f} ±{r['acc_std']:.4f}",
        "Acc Min"          : f"{min(r['accuracies']):.4f}",
        "Acc Max"          : f"{max(r['accuracies']):.4f}",
        "F1 Media (±Std)"  : f"{r['f1_media']:.4f} ±{r['f1_std']:.4f}",
        # ── conjunto de prueba principal ──
        "Acc Prueba"       : f"{acc_test:.4f}",
        "Precisión"        : f"{prec_test:.4f}",
        "Recall"           : f"{rec_test:.4f}",
        "F1 Prueba"        : f"{f1_test:.4f}",
        "TP"               : int(tp),
        "TN"               : int(tn),
        "FP"               : int(fp),
        "FN"               : int(fn),
        # ── grid search ──
        "Acc CV (train)"   : f"{grid.best_score_:.4f}",
        "Tiempo (s)"       : f"{tiempos[nombre]:.1f}",
        "Mejores Params"   : str(r["best_params"]),
    })

df_unificado = (pd.DataFrame(filas)
                  .sort_values("Acc Media (±Std)", ascending=False)
                  .reset_index(drop=True))
df_unificado.index += 1

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║         TABLA UNIFICADA DE MÉTRICAS — TODOS LOS MODELOS         ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print(df_unificado.to_string())
print()
print("Leyenda: Acc=Exactitud | Prec=Precisión | Rec=Recall | F1=F1-Score")
print("         TP=Verdadero Positivo | TN=Verdadero Negativo")
print("         FP=Falso Positivo     | FN=Falso Negativo")
print()
mejor_idx = df_unificado.index[0]
print(f"► MEJOR MODELO: {df_unificado.loc[mejor_idx, 'Modelo']}  "
      f"(Acc Media = {df_unificado.loc[mejor_idx, 'Acc Media (±Std)']})")


---
## 10. Árbol de decisión


In [ ]:
arbol_grid  = resultados_grid["Arbol de Decision"]
arbol_mejor = arbol_grid.best_estimator_
depth_real  = arbol_mejor.get_depth()

print("Arbol de Decision ajustado:")
print("  Profundidad real : " + str(depth_real))
print("  Numero de hojas  : " + str(arbol_mejor.get_n_leaves()))
print("  Mejores params   : " + str(arbol_grid.best_params_))

viz_depth = min(depth_real, 4)
plt.figure(figsize=(20, 8))
plot_tree(
    arbol_mejor,
    max_depth   = viz_depth,
    class_names = ["Negativo", "Positivo"],
    filled      = True,
    rounded     = True,
    fontsize    = 8,
    impurity    = False,
)
plt.title(
    "Arbol de Decision (primeros " + str(viz_depth) + " niveles de " + str(depth_real) + ")\n"
    "Naranja = Positivo (arroz), Azul = Negativo",
    fontsize=12, fontweight="bold"
)
plt.savefig("arbol_decision.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: arbol_decision.png")

---
## 11. Exportación del modelo

Se selecciona el modelo con mayor exactitud media y se reentrena con el 100% de los datos antes de exportar. Ver README para cómo usar el `.joblib` en inferencia.


In [ ]:
mejor_nombre = max(eval_resultados, key=lambda n: eval_resultados[n]["acc_media"])
mejor_r      = eval_resultados[mejor_nombre]
mejor_grid   = resultados_grid[mejor_nombre]

print("MODELO GANADOR:", mejor_nombre)
print("  (Ver Tabla Unificada de Métricas para el detalle completo)")
print()
print("Reentrenando", mejor_nombre, "con el 100% de los datos...")
modelo_final = clone(mejor_grid.best_estimator_)
modelo_final.fit(X, y)
print("Listo.")


In [ ]:
nombre_archivo = CARNE + "_" + NOMBRE + "_" + APELLIDO + ".joblib"
nombre_archivo_export = nombre_archivo
joblib.dump(modelo_final, nombre_archivo)

print("Modelo exportado : " + nombre_archivo)
print("Algoritmo        : " + type(modelo_final).__name__)

# Verificación rápida
modelo_cargado = joblib.load(nombre_archivo)
y_verif = modelo_cargado.predict(X[:3])
print("Verificacion (primeras 3 filas):")
print("  Predicciones : " + str(y_verif.tolist()))
print("  Reales       : " + str(y[:3].tolist()))

# Descargar desde Colab
files.download(nombre_archivo)
print("Descarga iniciada: " + nombre_archivo)

---
## 12. Resumen final


In [ ]:
nombre_archivo_export = CARNE + "_" + NOMBRE + "_" + APELLIDO + ".joblib"

print("=" * 65)
print("  RESUMEN FINAL — PROYECTO 1")
print("  Clasificación de Contaminaciones en Línea de Producción")
print("=" * 65)
print("  Estudiante  :", NOMBRE, APELLIDO, "(" + CARNE + ")")
print("  Dataset     :", len(X), "imágenes x", X.shape[1], "píxeles (128×128)")
print("  Clases      :", (y==1).sum(), "positivos,", (y==0).sum(), "negativos")
print("  Split       :", str(int((1-TEST_SIZE)*100)) + "/" + str(int(TEST_SIZE*100)),
      "— Evaluación repetida", str(N_REPETICIONES) + "x")
print()
print("  ► Ver sección 9 para la Tabla Unificada con TODAS las métricas")
print()
print("  MODELO EXPORTADO :", mejor_nombre)
print("  Hiperparámetros  :", mejor_r["best_params"])
print("  Archivo          :", nombre_archivo_export)
print("=" * 65)
